## Worldwide Earthquake Events API - Gold Layer Processing

In [1]:
from datetime import datetime

# Start time
start_time = datetime.now()

StatementMeta(, 5375d259-967e-4568-9170-be46b5b0e79b, 5, Finished, Available, Finished, False)

In [2]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType
import reverse_geocoder as rg

StatementMeta(, 5375d259-967e-4568-9170-be46b5b0e79b, 6, Finished, Available, Finished, False)

In [3]:
"""
Reference Snippet: Loading the earthquake JSON file based on a sample start_date

(Keep this snippet commented when running the pipeline)
"""

# from datetime import date, timedelta

# start_date = date.today() - timedelta(7) 

StatementMeta(, 5375d259-967e-4568-9170-be46b5b0e79b, 7, Finished, Available, Finished, False)

In [4]:
df = spark.read.table("earthquake_events_silver").filter(col('time') > start_date)

StatementMeta(, 5375d259-967e-4568-9170-be46b5b0e79b, 8, Finished, Available, Finished, False)

In [5]:
"""
Reference Snippet: Verifying reverse_geocoder output using sample earthquake coordinates

(Keep this snippet commented when running the pipeline)
"""

# temp_df = spark.read.table("earthquake_events_silver")
# display(temp_df.limit(2))


# # Selecting first ID's latitude and longitude as coordinates
# coordinates1 = (35.9894981384277, -120.546501159668)

# # Display the location using coordinates
# print("Location 1 : ", rg.search(coordinates1))

# # Selecting second ID's latitude and longitude as coordinates
# coordinates2 = (32.3769, 142.4944)

# # Display the location using coordinates
# print("Location 2 : ", rg.search(coordinates2))

# # To get the country code from Location 2 array, we do the following
# # [0] because there is one element in the array, ['cc'] because the country code can be accessed using that key
# print("Location 2 - Country Code: ", rg.search(coordinates2)[0]['cc']) 

StatementMeta(, 5375d259-967e-4568-9170-be46b5b0e79b, 9, Finished, Available, Finished, False)

'\nReference Snippet: Verifying reverse_geocoder output using sample earthquake coordinates\n\n(Keep this snippet commented when running the pipeline)\n'

In [6]:
def get_country_code(lat, lon):
    """
    Retrieve the country code for a given latitude and longitude.

    Parameters:
    lat (float or str): Latitude of the location.
    lon (float or str): Longitude of the location.

    Returns:
    str: Country code of the location, retrieved using the reverse geocoding API.

    Example:
    >>> get_country_details(48.8588443, 2.2943506)
    'FR'
    """
    coordinates = (float(lat), float(lon))
    return rg.search(coordinates)[0]['cc']

StatementMeta(, 5375d259-967e-4568-9170-be46b5b0e79b, 10, Finished, Available, Finished, False)

In [7]:
# registering the udfs so they can be used on spark dataframes
get_country_code_udf = udf(get_country_code, StringType())

StatementMeta(, 5375d259-967e-4568-9170-be46b5b0e79b, 11, Finished, Available, Finished, False)

In [8]:
# adding country_code and city attributes
df_with_location = df.withColumn("country_code", get_country_code_udf(col("latitude"), col("longitude")))

StatementMeta(, 5375d259-967e-4568-9170-be46b5b0e79b, 12, Finished, Available, Finished, False)

In [9]:
# adding significance classification
df_with_location_sig_class = df_with_location.\
                                withColumn('sig_class', 
                                            when(col("sig") < 100, "Low").\
                                            when((col("sig") >= 100) & (col("sig") < 500), "Moderate").\
                                            otherwise("High")
                                            )

StatementMeta(, 5375d259-967e-4568-9170-be46b5b0e79b, 13, Finished, Available, Finished, False)

In [10]:
# appending the data to the gold table
df_with_location_sig_class.write.mode('append').saveAsTable('earthquake_events_gold')

StatementMeta(, 5375d259-967e-4568-9170-be46b5b0e79b, 14, Finished, Available, Finished, False)

In [11]:
end_time = datetime.now()

# Runtime difference
runtime = end_time - start_time

print("Runtime    :", runtime)

StatementMeta(, 5375d259-967e-4568-9170-be46b5b0e79b, 15, Finished, Available, Finished, False)

Runtime    : 0:04:41.317365
